In [1]:
# thinkpython.py content
import contextlib
import io
import re

def extract_function_name(text):
    """Find a function definition and return its name.
     text: String
     returns: String or None
     """
    pattern = r"def\s+(\w+)\s*\("
    match = re.search(pattern, text)
    if match:
        func_name = match.group(1)
        return func_name
    else:
        return None

# the functions that define cell magic commands are only defined
# if we're running in Jupyter.
try:
    from IPython.core.magic import register_cell_magic
    from IPython.core.magic_arguments import argument, magic_arguments, parse_argstring

    @register_cell_magic
    def add_method_to(args, cell):

        # get the name of the function defined in this cell
        func_name = extract_function_name(cell)
        if func_name is None:
            return f"This cell doesn't define any new functions."

        # get the class we're adding it to
        namespace = get_ipython().user_ns
        class_name = args
        cls = namespace.get(class_name, None)
        if cls is None:
            return f"Class '{class_name}' not found."

        # save the old version of the function if it was already defined
        old_func = namespace.get(func_name, None)
        if old_func is not None:
            del namespace[func_name]

        # Execute the cell to define the function
        get_ipython().run_cell(cell)

        # get the newly defined function
        new_func = namespace.get(func_name, None)
        if new_func is None:
            return f"This cell didn't define {func_name}."

        # add the function to the class and remove it from the namespace
        setattr(cls, func_name, new_func)
        del namespace[func_name]

        # restore the old function to the namespace
        if old_func is not None:
            namespace[func_name] = old_func

    @register_cell_magic
    def expect_error(line, cell):
        try:
            get_ipython().run_cell(cell)
        except Exception as e:
            get_ipython().run_cell("%tb")

    @magic_arguments()
    @argument("exception", help="Type of exception to catch")
    @register_cell_magic
    def expect(line, cell):
        args = parse_argstring(expect, line)
        exception = eval(args.exception)
        try:
            get_ipython().run_cell(cell)
        except exception as e:
            get_ipython().run_cell("%tb")

    def traceback(mode):
        """Set the traceback mode.
         mode: string
         """
        with contextlib.redirect_stdout(io.StringIO()):
            get_ipython().run_cell(f"%xmode {mode}")

    traceback("Minimal")
except (ImportError, NameError):
    print("Warning: IPython is not available, cell magic not defined.")


# 17. Inheritance
Run this notebook on Colab

The language feature most often associated with object-oriented programming is inheritance. Inheritance is the ability to define a new class that is a modified version of an existing class. In this chapter I demonstrate inheritance using classes that represent playing cards, decks of cards, and poker hands. If you don’t play poker, don’t worry – I’ll tell you what you need to know.

17.1. Representing cards
There are 52 playing cards in a standard deck – each of them belongs to one of four suits and one of thirteen ranks. The suits are Spades, Hearts, Diamonds, and Clubs. The ranks are Ace, 2, 3, 4, 5, 6, 7, 8, 9, 10, Jack, Queen, and King. Depending on which game you are playing, an Ace can be higher than King or lower than 2.

If we want to define a new object to represent a playing card, it is obvious what the attributes should be: rank and suit. It is less obvious what type the attributes should be. One possibility is to use strings like 'Spade' for suits and 'Queen' for ranks. A problem with this implementation is that it would not be easy to compare cards to see which had a higher rank or suit.

An alternative is to use integers to encode the ranks and suits. In this context, “encode” means that we are going to define a mapping between numbers and suits, or between numbers and ranks. This kind of encoding is not meant to be a secret (that would be “encryption”).

With this encoding, we can compare suits by comparing their codes.

To encode the ranks, we’ll use the integer 2 to represent the rank 2, 3 to represent 3, and so on up to 10. The following table shows the codes for the face cards.

And we can use either 1 or 14 to represent an Ace, depending on whether we want it to be considered lower or higher than the other ranks.

To represent these encodings, we will use two lists of strings, one with the names of the suits and the other with the names of the ranks.

Here’s a definition for a class that represents a playing card, with these lists of strings as class variables, which are variables defined inside a class definition, but not inside a method.

In [2]:
class Card:
    """Represents a standard playing card."""

    suit_names = ['Clubs', 'Diamonds', 'Hearts', 'Spades']
    rank_names = [None, 'Ace', '2', '3', '4', '5', '6', '7',
                  '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']

In [3]:
Card.suit_names

['Clubs', 'Diamonds', 'Hearts', 'Spades']

In [4]:
Card.suit_names[0]

'Clubs'

In [5]:
Card.rank_names[11]

'Jack'

## 17.2. Card attributes
Here’s an __init__ method for the Card class – it takes suit and rank as parameters and assigns them to attributes with the same names.

In [6]:
%%add_method_to Card

def __init__(self, suit, rank):
    self.suit = suit
    self.rank = rank

In [7]:
queen = Card(1, 12)

In [8]:
queen.suit, queen.rank

(1, 12)

In [9]:
queen.suit_names

['Clubs', 'Diamonds', 'Hearts', 'Spades']

## 17.3. Printing cards
Here’s a __str__ method for Card objects.

In [10]:
%%add_method_to Card

def __str__(self):
    rank_name = Card.rank_names[self.rank]
    suit_name = Card.suit_names[self.suit]
    return f'{rank_name} of {suit_name}'

In [11]:
print(queen)

Queen of Diamonds


## 17.4. Comparing cards
Suppose we create a second Card object with the same suit and rank.

In [12]:
queen2 = Card(1, 12)
print(queen2)

Queen of Diamonds


In [13]:
# If we use the == operator to compare them, it checks whether queen and queen2 refer to the same object.
queen == queen2  # They don’t, so it returns False. We can change this behavior by defining the special method __eq__.

False

In [14]:
%%add_method_to Card

def __eq__(self, other):
    return self.suit == other.suit and self.rank == other.rank

In [15]:
queen == queen2

True

In [16]:
six = Card(1, 6)
print(six)

6 of Diamonds


In [17]:
queen == six

False

#If we use the != operator, Python invokes a special method called __ne__, if it exists. Otherwise it invokes__eq__ and inverts the result – so if __eq__ returns True, the result of the != operator is False.

In [18]:
queen != queen2

False

In [19]:
queen != six

True

In [20]:
# Now suppose we want to compare two cards to see which is bigger. If we use one of the relational operators, we get a TypeError.
# queen < queen2
# TypeError: '<' not supported between instances of 'Card' and 'Card'

##### To change the behavior of the < operator, we can define a special method called __lt__, which is short for “less than”. For the sake of this example, let’s assume that suit is more important than rank – so all Spades outrank all Hearts, which outrank all Diamonds, and so on. If two cards have the same suit, the one with the higher rank wins.

##### To implement this logic, we’ll use the following method, which returns a tuple containing a card’s suit and rank, in that order.

In [21]:
%%add_method_to Card

def to_tuple(self):
    return (self.suit, self.rank)

In [22]:
%%add_method_to Card

def __lt__(self, other):
    return self.to_tuple() < other.to_tuple()

In [23]:
six < queen

True

In [24]:
# If we use the > operator, it invokes a special method called __gt__, if it exists. Otherwise it invokes __lt__ with the arguments in the opposite order.
queen < queen2

False

In [25]:
queen > queen2

False

In [26]:
# Finally, if we use the <= operator, it invokes a special method called __le__.

In [27]:
%%add_method_to Card

def __le__(self, other):
    return self.to_tuple() <= other.to_tuple()

In [28]:
queen <= queen2

True

In [29]:
queen <= six

False

In [30]:
# If we use the >= operator, it uses __ge__ if it exists. Otherwise, it invokes __le__ with the arguments in the opposite order.
queen >= six

True

As we have defined them, these methods are complete in the sense that we can compare any two Card objects, and consistent in the sense that results from different operators don’t contradict each other. With these two properties, we can say that Card objects are totally ordered. And that means, as we’ll see soon, that they can be sorted.

## 17.5. Decks
Now that we have objects that represent cards, let’s define objects that represent decks. The following is a class definition for Deck with an __init__ method takes a list of Card objects as a parameter and assigns it to an attribute called cards.

In [31]:
class Deck:

    def __init__(self, cards):
        self.cards = cards

In [32]:
%%add_method_to Deck
# To create a list that contains the 52 cards in a standard deck, we’ll use the following static method.

def make_cards():
    cards = []
    for suit in range(4):
        for rank in range(2, 15):
            card = Card(suit, rank)
            cards.append(card)
    return cards

In [33]:
cards = Deck.make_cards()
deck = Deck(cards)
len(deck.cards)

52

In [34]:
# 17.6. Printing the deck
# Here is a __str__ method for Deck.

In [35]:
%%add_method_to Deck

def __str__(self):
    res = []
    for card in self.cards:
        res.append(str(card))
    return '\n'.join(res)

In [36]:
small_deck = Deck([queen, six])

In [37]:
str(small_deck)

'Queen of Diamonds\n6 of Diamonds'

##### When Jupyter displays a string, it shows the “representational” form of the string, which represents a newline with the sequence \n. However, if we print the result, Jupyter shows the “printable” form of the string, which prints the newline as whitespace.

In [38]:
print(small_deck)

Queen of Diamonds
6 of Diamonds


## 17.7. Add, remove, shuffle and sort

In [39]:
%%add_method_to Deck

# To deal cards, we would like a method that removes a card from the deck and returns it. The list method pop provides a convenient way to do that.
def take_card(self):
    return self.cards.pop()

In [40]:
card = deck.take_card()
print(card)

Ace of Spades


In [41]:
len(deck.cards)

51

In [42]:
%%add_method_to Deck

def put_card(self, card):
    self.cards.append(card)

In [43]:
deck.put_card(card)
len(deck.cards)

52

In [44]:
import random

In [45]:
%%add_method_to Deck

def shuffle(self):
    random.shuffle(self.cards)

In [46]:
deck.shuffle()
for card in deck.cards[:4]:
    print(card)

3 of Clubs
10 of Clubs
2 of Hearts
Ace of Diamonds


In [47]:
%%add_method_to Deck

def sort(self):
    self.cards.sort()

In [48]:
deck.sort()

In [49]:
for card in deck.cards[:4]:
    print(card)

2 of Clubs
3 of Clubs
4 of Clubs
5 of Clubs


## 17.8. Parents and children

In [50]:
class Hand(Deck):
    """Represents a hand of playing cards."""

In [51]:
%%add_method_to Hand

def __init__(self, label=''):
    self.label = label
    self.cards = []

In [52]:
hand = Hand('player 1')
hand.label

'player 1'

In [53]:
deck = Deck(cards)
card = deck.take_card()
hand.put_card(card)
print(hand)

Ace of Spades


In [54]:
%%add_method_to Deck

def move_cards(self, other, num):
    for i in range(num):
        card = self.take_card()
        other.put_card(card)

## 17.9. Specialization

In [55]:
class BridgeHand(Hand):
    """Represents a bridge hand."""

    hcp_dict = {
        'Ace': 4,
        'King': 3,
        'Queen': 2,
        'Jack': 1,
    }

In [56]:
rank = 12
rank_name = Card.rank_names[rank]
score = BridgeHand.hcp_dict.get(rank_name, 0)
rank_name, score

('Queen', 2)

In [57]:
%%add_method_to BridgeHand

def high_card_point_count(self):
    count = 0
    for card in self.cards:
        rank_name = Card.rank_names[card.rank]
        count += BridgeHand.hcp_dict.get(rank_name, 0)
    return count

In [58]:
hand = BridgeHand('player 2')

deck.shuffle()
deck.move_cards(hand, 5)
print(hand)

3 of Hearts
7 of Diamonds
5 of Spades
6 of Diamonds
2 of Clubs


In [59]:
hand.high_card_point_count()

0

## 17.10. Debugging

In [60]:
def find_defining_class(obj, method_name):
    """Find the class where the given method is defined."""
    print(type(obj).mro())
    for typ in type(obj).mro():
        if method_name in vars(typ):
            print(vars(typ))
            return typ
    return f'Method {method_name} not found.'

In [61]:
hand = BridgeHand('player 3')
find_defining_class(hand, 'shuffle')

[<class '__main__.BridgeHand'>, <class '__main__.Hand'>, <class '__main__.Deck'>, <class 'object'>]
{'__module__': '__main__', '__init__': <function Deck.__init__ at 0x1085be660>, '__dict__': <attribute '__dict__' of 'Deck' objects>, '__weakref__': <attribute '__weakref__' of 'Deck' objects>, '__doc__': None, 'make_cards': <function make_cards at 0x1085bea20>, '__str__': <function __str__ at 0x1085bed40>, 'take_card': <function take_card at 0x1085beca0>, 'put_card': <function put_card at 0x1085bf1a0>, 'shuffle': <function shuffle at 0x1085bf2e0>, 'sort': <function sort at 0x1085bf060>, 'move_cards': <function move_cards at 0x1085bf420>}


__main__.Deck

In [62]:
"""17.11. Glossary
inheritance: The ability to define a new class that is a modified version of a previously defined class.

encode: To represent one set of values using another set of values by constructing a mapping between them.

class variable: A variable defined inside a class definition, but not inside any method.

totally ordered: A set of objects is totally ordered if we can compare any two elements and the results are consistent.

delegation: When one method passes responsibility to another method to do most or all of the work.

parent class: A class that is inherited from.

child class: A class that inherits from another class.

specialization: A way of using inheritance to create a new class that is a specialized version of an existing class."""

'17.11. Glossary\ninheritance: The ability to define a new class that is a modified version of a previously defined class.\n\nencode: To represent one set of values using another set of values by constructing a mapping between them.\n\nclass variable: A variable defined inside a class definition, but not inside any method.\n\ntotally ordered: A set of objects is totally ordered if we can compare any two elements and the results are consistent.\n\ndelegation: When one method passes responsibility to another method to do most or all of the work.\n\nparent class: A class that is inherited from.\n\nchild class: A class that inherits from another class.\n\nspecialization: A way of using inheritance to create a new class that is a specialized version of an existing class.'

## 17.12. Exercises

In [63]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


In [64]:
"""17.12.1. Ask a Virtual Assistant
When it goes well, object-oriented programming can make programs more readable, testable, and reusable. But it can also make programs complicated and hard to maintain. As a result, OOP is a topic of controversy – some people love it, and some people don’t.

To learn more about the topic, ask a virtual assistant:

What are some pros and cons of object-oriented programming?

What does it mean when people say “favor composition over inheritance”?

What is the Liskov substitution principle?

Is Python an object-oriented language?

What are the requirements for a set to be totally ordered?

And as always, consider using a virtual assistant to help with the following exercises.
"""

'17.12.1. Ask a Virtual Assistant\nWhen it goes well, object-oriented programming can make programs more readable, testable, and reusable. But it can also make programs complicated and hard to maintain. As a result, OOP is a topic of controversy – some people love it, and some people don’t.\n\nTo learn more about the topic, ask a virtual assistant:\n\nWhat are some pros and cons of object-oriented programming?\n\nWhat does it mean when people say “favor composition over inheritance”?\n\nWhat is the Liskov substitution principle?\n\nIs Python an object-oriented language?\n\nWhat are the requirements for a set to be totally ordered?\n\nAnd as always, consider using a virtual assistant to help with the following exercises.\n'

## 17.12.1. Ask a Virtual Assistant

### 1. Pros and cons of OOP

**Pros:**
- Organization: groups related data and methods into classes.
- Reusability: create multiple objects from one class.
- Maintainability: makes well-organized code easier to modify.
- Encapsulation: hides implementation details behind an interface.

**Cons:**
- Complexity: too many classes can make code difficult to understand.
- Overengineering: simple problems may not need classes.
- Inheritance issues: subclasses can become tightly coupled to parent classes.
- Debugging: behavior may be spread across multiple classes and methods.

### 2. Favor composition over inheritance

Prefer building classes using other objects when this is more flexible than inheritance.

- **Inheritance (“is a”):** A `Kangaroo` is an `Animal`.
- **Composition (“has a”):** A `Kangaroo` has a `Pouch`.

Composition can make programs more flexible because components can be replaced independently.

### 3. Liskov substitution principle (LSP)

Objects of a subclass should be usable wherever objects of the parent class are expected, without breaking the program's intended behavior.

Example: a `Kangaroo` subclass of `Animal` should satisfy the behavioral expectations of an `Animal`.

### 4. Is Python an object-oriented language?

Yes. Python supports classes, objects, inheritance, encapsulation, and polymorphism.

Python is also a **multi-paradigm language**, supporting object-oriented, procedural, and functional programming.

### 5. Requirements for a total order

A set is totally ordered under a relation if the relation satisfies:

- **Reflexivity:** \(a \leq a\).
- **Antisymmetry:** if \(a \leq b\) and \(b \leq a\), then \(a=b\).
- **Transitivity:** if \(a \leq b\) and \(b \leq c\), then \(a \leq c\).
- **Comparability:** for any \(a\) and \(b\), either \(a \leq b\) or \(b \leq a\).

Example: integers under the usual \(\leq\) relation are totally ordered. Sets under the subset relation are not, because some pairs of sets are incomparable.

### 17.12.2. Exercise
In contract bridge, a “trick” is a round of play in which each of four players plays one card. To represent those cards, we’ll define a class that inherits from Deck.

In [65]:
class Trick(Deck):
    """Represents a trick in contract bridge."""


As an example, consider this trick, where the first player leads with the 3 of Diamonds, which means that Diamonds are the “led suit”. The second and third players “follow suit”, which means they play a card with the led suit. The fourth player plays a card of a different suit, which means they cannot win the trick. So the winner of this trick is the third player, because they played the highest card in the led suit.

In [66]:
cards = [Card(1, 3),
         Card(1, 10),
         Card(1, 12),
         Card(2, 13)]
trick = Trick(cards)
print(trick)

3 of Diamonds
10 of Diamonds
Queen of Diamonds
King of Hearts


##### Write a Trick method called find_winner that loops through the cards in the Trick and returns the index of the card that wins. In the previous example, the index of the winning card is 2.

In [67]:
%%add_method_to Trick

def find_winner(self):
    lead_suit = self.cards[0].suit
    lead_rank = self.cards[0].rank
    winner_index = 0
    for n, card in enumerate(self.cards):
        if card.suit == lead_suit and card.rank > lead_rank:
            lead_rank = card.rank
            winner_index = n
    return winner_index

In [68]:
trick.find_winner()

2

In [69]:
### 17.12.3. Exercise

##### The next few exercises ask you to write functions that classify poker hands. If you are not familiar with poker, I’ll explain what you need to know. We’ll use the following class to represent poker hands.

In [70]:
class PokerHand(Hand):
    """Represents a poker hand."""

    def get_suit_counts(self):
        counter = {}
        for card in self.cards:
            key = card.suit
            counter[key] = counter.get(key, 0) + 1
        return counter

    def get_rank_counts(self):
        counter = {}
        for card in self.cards:
            key = card.rank
            counter[key] = counter.get(key, 0) + 1
        return counter

##### As a first exercise, we’ll write a method called has_flush that checks whether a hand has a “flush” – that is, whether it contains at least five cards of the same suit. In most varieties of poker, a hand contains either five or seven cards, but there are some exotic variations where a hand contains other numbers of cards. But regardless of how many cards there are in a hand, the only ones that count are the five that make the best hand.

In [71]:
%%add_method_to PokerHand
def has_flush(self):
    suit_counts = self.get_suit_counts()
    for count in suit_counts.values():
        if count > 4:
            return True
    return False

### 17.12.4. Exercise
Write a method called has_straight that checks whether a hand contains a straight, which is a set of five cards with consecutive ranks. For example, if a hand contains ranks 5, 6, 7, 8, and 9, it contains a straight.

An Ace can come before a two or after a King, so Ace, 2, 3, 4, 5 is a straight and so is 10, Jack, Queen, King, Ace. But a straight cannot “wrap around”, so King, Ace, 2, 3, 4 is not a straight.

In [72]:
%%add_method_to PokerHand
def has_straight(self):
    rank_count = self.get_rank_counts()
    ranks = []  # ranks in number
    for rank in rank_count:
        ranks.append(rank)

    ranks.sort()
    ranks_names = []
    for rank in ranks:
        ranks_names.append(Card.rank_names[rank])

    # Checks if it is a flush
    if ranks_names in Card.rank_names:
        return True
    return False

### 17.12.5. Exercise
A hand has a straight flush if it contains a set of five cards that are both a straight and a flush – that is, five cards of the same suit with consecutive ranks. Write a PokerHand method that checks whether a hand has a straight flush.

In [73]:
%%add_method_to PokerHand
def has_straight_flush(self):
    return self.has_straight() and self.has_flush()

### 17.12.6. Exercise
A poker hand has a pair if it contains two or more cards with the same rank. Write a PokerHand method that checks whether a hand contains a pair.

To test your method, here’s a hand that has a pair.

In [74]:
%%add_method_to PokerHand
def has_pair(self):
    rank_counts = self.get_rank_counts()
    for rank in rank_counts:
        if rank_counts[rank] > 1:
            return True
    return False

In [75]:
# from copy import deepcopy
# pair = deepcopy(bad_hand)
# pair.put_card(Card(1, 2))
# print(pair)

# ['Clubs', 'Diamonds', 'Hearts', 'Spades']
# pair = [
#     "2 of Clubs",  # Card(suit, rank) Card(0,2)
#     "3 of Clubs",  # Card(0,3)
#     "4 of Hearts",  # Card(2,4)
#     "5 of Spades",  # Card(3,5)
#     "7 of Clubs",  # Card(0,7)
#     "2 of Diamonds"  # Card(1,2)
# ]

cards = [Card(0,2),Card(0,3),Card(2,4),Card(3,5),Card(0,7),Card(1,2)]

pair = PokerHand("pair")

for card in cards:
    pair.put_card(card)

In [76]:
print(pair)

2 of Clubs
3 of Clubs
4 of Hearts
5 of Spades
7 of Clubs
2 of Diamonds


In [77]:
pair.has_pair()    # should return True

True

In [78]:
bad_hand = PokerHand('bad_hand')
for card in cards[:-1]:
    bad_hand.put_card(card)
print(bad_hand)

2 of Clubs
3 of Clubs
4 of Hearts
5 of Spades
7 of Clubs


In [79]:
bad_hand.has_pair()    # should return False

False

### 17.12.7. Exercise
A hand has a full house if it contains three cards of one rank and two cards of another rank. Write a PokerHand method that checks whether a hand has a full house.

In [80]:
%%add_method_to PokerHand
def has_full_house(self):
    rank_counts = self.get_rank_counts()
    counts = {}  # rank counts with rank appear more than once
    for rank in rank_counts:
        if rank_counts[rank] > 1:
            counts[rank] = rank_counts[rank]

    if len(counts) > 1:  # only if ranks with more than one counts appears more than once
        for count in counts.values():
            if count > 2:  # only if one of the rank counts is more than two
                return True

    return False

### 17.12.8. Exercise
This exercise is a cautionary tale about a common error that can be difficult to debug. Consider the following class definition.(A cautionary tale is a story that teaches people a lesson by showing what can go wrong if they make a particular mistake or behave in a certain way.)

In [103]:
class Kangaroo:
    """A Kangaroo is a marsupial."""

    # def __init__(self, name, contents=[]):  # In Python, a default argument such as contents=[] is created once when the function is defined, not every time __init__ is called.
    #
    #     """Initialize the pouch contents.
    #
    #     name: string
    #     contents: initial pouch contents.
    #     """
    #     self.name = name
    #     self.contents = contents

    def __init__(self, name, contents=None):
        self.name = name

        if contents is None:
            contents = []

        self.contents = contents

    def __str__(self):
        """Return a string representaion of this Kangaroo.
        """
        t = [ self.name + ' has pouch contents:' ]
        for obj in self.contents:
            s = '    ' + object.__str__(obj)
            t.append(s)
        return '\n'.join(t)

    def put_in_pouch(self, item):
        """Adds a new item to the pouch contents.

        item: object to be added
        """
        self.contents.append(item)


In [104]:
kanga = Kangaroo('Kanga')
roo = Kangaroo('Roo')

In [105]:
print(roo)

Roo has pouch contents:


In [106]:
roo.contents

[]

In [107]:
kanga.put_in_pouch('wallet')
kanga.put_in_pouch('car keys')
kanga.put_in_pouch(roo)

In [108]:
print(kanga)

Kanga has pouch contents:
    'wallet'
    'car keys'


In [109]:
print(roo)

Roo has pouch contents:


##### Roo’s pouch contains the same contents as Kanga’s, including a reference to roo!

See if you can figure out what went wrong. Then ask a virtual assistant, “What’s wrong with the following program?” and paste in the definition of Kangaroo.

    def __init__(self, name, contents=[]):  # In Python, a default argument such as contents=[] is created once when the function is defined, not every time __init__ is called.